<a href="https://colab.research.google.com/github/suryaph971/Langchain/blob/main/QA_on_private_documents(RAG)_with_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install -r requirements\ \(1\).txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.9 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=3579d7134339a000e9e7b602c37ec3781210e6b3bddc6ceff80810d051c73763
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langcha

In [3]:
import os
from dotenv import find_dotenv,load_dotenv
load_dotenv(find_dotenv(),override=True)

True

In [4]:
def load_document(file):
    import os
    name,extension = os.path.splitext(file)
    if extension == '.pdf':
        from langchain.document_loaders import PyPDFLoader
        print(f'Loading {file}')
        loader = PyPDFLoader(file)
    elif extension == '.docx':
        from langchain.document_loaders import Docx2txtLoader
        print(f'Loading {file}')
        loader = Docx2txtLoader(file)
    elif extension == '.txt':
        from langchain.document_loaders import TextLoader
        loader = TextLoader(file)
    else:
        print('Document type is not supported')
        return None

    data = loader.load()
    return data


In [5]:
#wikipedia
def load_from_wikipedia(query,lang='en',load_max_docs=2):
    from langchain.document_loaders import WikipediaLoader
    loader = WikipediaLoader(query=query,lang=lang,load_max_docs=load_max_docs)
    data = loader.load()
    return data

In [6]:
def chunk_data(data,chunk_size=256):
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size,chunk_overlap=0)
    chunks = text_splitter.split_documents(data)
    return chunks

In [8]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 19.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [9]:
pip install -q langchain_google_genai


In [10]:
pip install -q chromadb

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [11]:
def index_or_fetch_embeddings_chroma(chunks,persist_directory='./chroma_db'):
    from langchain.vectorstores import Chroma
    from langchain_google_genai import GoogleGenerativeAIEmbeddings

    embeddings = GoogleGenerativeAIEmbeddings(model='text-embedding-004',dimensions=1536)
    vector_store = Chroma.from_documents(chunks,embeddings,persist_directory=persist_directory)
    return vector_store

In [12]:
def load_embeddings_chroma(persist_directory='./chroma_db'):
    from langchain.vectorstores import Chroma
    from langchain_google_genai import GoogleGenerativeAIEmbeddings

    embeddings = GoogleGenerativeAIEmbeddings(model='text-embedding-004',dimensions=1536)
    vector_store = Chroma(persist_directory=persist_directory,embedding_function=embeddings)
    return vector_store

In [15]:
data = load_document('us_constitution.pdf')
chunks = chunk_data(data)
print(len(chunks))
vector_store = index_or_fetch_embeddings_chroma(chunks)

Loading us_constitution.pdf
224


In [16]:
from re import search
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-1.5-flash')
retriever = vector_store.as_retriever(search_type='similarity',search_kwargs={'k':5})
memory = ConversationBufferMemory(memory_key='chat_history',return_messages=True)
crc = ConversationalRetrievalChain.from_llm(llm=llm,chain_type='stuff',retriever=retriever,memory=memory,verbose=False)


/tmp/ipython-input-4142181150.py:8: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key='chat_history',return_messages=True)


In [17]:
def ask_question(q, chain):
    result = chain.invoke({'question': q})
    return result

In [18]:
import time
i=1
print('Write Quit or Exit to quit')
while True:
    q=input(f'Question #{i}')
    i=i+1
    if q.lower() in ['quit','exit']:
        print("Exiting")
        time.sleep(2)
        break
    else:
        answer = ask_question(q,crc)
        print(f'\nAnswer: {answer}')
        print(f'\n {"-" * 50} \n')

Write Quit or Exit to quit
Question #1What is the first amendment described in the document?

Answer: {'question': 'What is the first amendment described in the document?', 'chat_history': [HumanMessage(content='What is the first amendment described in the document?', additional_kwargs={}, response_metadata={}), AIMessage(content='The First Amendment states that Congress shall make no law respecting an establishment of religion, or prohibiting the free exercise thereof; or abridging the freedom of speech, or of the press; or the right of the people peaceably to assemble, and to petition the Government for a redress of grievances.', additional_kwargs={}, response_metadata={})], 'answer': 'The First Amendment states that Congress shall make no law respecting an establishment of religion, or prohibiting the free exercise thereof; or abridging the freedom of speech, or of the press; or the right of the people peaceably to assemble, and to petition the Government for a redress of grievances